In [5]:
import os
import cv2
import pandas as pd
import numpy as np
import math

# --- Configuration ---
DATA_DIR = "data"
IMAGES_DIR = "images"
CSV_PATH = "annotations.csv"

YOLO_LABELS_DIR = os.path.join(DATA_DIR, "yolo_labels")
CROPS_DIR = os.path.join(DATA_DIR, "crops")
os.makedirs(YOLO_LABELS_DIR, exist_ok=True)
os.makedirs(CROPS_DIR, exist_ok=True)

df = pd.read_csv(CSV_PATH)

crop_data = []

for idx, row in df.iterrows():
    img_name = row['image']
    # Note: Ensure extensions match your actual files (PNG vs JPG)
    if not img_name.endswith('.jpg') and not img_name.endswith('.png'):
        continue 
        
    img_path = os.path.join(IMAGES_DIR, img_name)
    img = cv2.imread(img_path)
    if img is None:
        continue
    
    h, w, _ = img.shape
    
    # 1. Generate YOLO Bounding Box Format
    # YOLO format: class x_center y_center width height (normalized 0 to 1)
    x_center = row['center_x'] / w
    y_center = row['center_y'] / h
    box_w = row['bbox_w'] / w
    box_h = row['bbox_h'] / h
    
    label_path = os.path.join(YOLO_LABELS_DIR, img_name.rsplit('.', 1)[0] + '.txt')
    with open(label_path, 'a') as f:
        f.write(f"0 {x_center} {y_center} {box_w} {box_h}\n")
        
    # 2. Extract Crops for Angle Regression
    # We pad the crop slightly to ensure the whole tube is visible regardless of rotation
    pad = 10
    x1 = max(0, int(row['center_x'] - row['bbox_w']/2 - pad))
    y1 = max(0, int(row['center_y'] - row['bbox_h']/2 - pad))
    x2 = min(w, int(row['center_x'] + row['bbox_w']/2 + pad))
    y2 = min(h, int(row['center_y'] + row['bbox_h']/2 + pad))
    
    crop = img[y1:y2, x1:x2]
    crop_filename = f"crop_{idx}.jpg"
    cv2.imwrite(os.path.join(CROPS_DIR, crop_filename), crop)
    
    # Save crop mapping for PyTorch Dataset
    angle = row['angle_deg']
    # Convert angle to radians for sin/cos
    angle_rad = math.radians(angle)
    crop_data.append({
        'crop_name': crop_filename,
        'angle_deg': angle,
        'cos_theta': math.cos(angle_rad),
        'sin_theta': math.sin(angle_rad)
    })

crop_df = pd.DataFrame(crop_data)
crop_df.to_csv(os.path.join(DATA_DIR, 'crop_annotations.csv'), index=False)
print("Data preparation complete.")

Data preparation complete.


In [6]:
from ultralytics import YOLO
import os
import shutil
import yaml

# 1. Define paths
DATA_DIR = "data"
IMAGES_DIR = "images"
LABELS_DIR = os.path.join(DATA_DIR, "yolo_labels") 

# YOLOv8 expects an 'images' and 'labels' directory structure
YOLO_IMAGES_DIR = os.path.join(DATA_DIR, "images", "train")
YOLO_LABELS_DIR = os.path.join(DATA_DIR, "labels", "train")

os.makedirs(YOLO_IMAGES_DIR, exist_ok=True)
os.makedirs(YOLO_LABELS_DIR, exist_ok=True)

# 2. Move images and labels into the YOLO format structure
print("Organizing files for YOLO...")
for file in os.listdir(IMAGES_DIR):
    if file.endswith('.jpg') or file.endswith('.png'):
        shutil.copy(os.path.join(IMAGES_DIR, file), os.path.join(YOLO_IMAGES_DIR, file))

for file in os.listdir(LABELS_DIR):
    if file.endswith('.txt'):
        shutil.copy(os.path.join(LABELS_DIR, file), os.path.join(YOLO_LABELS_DIR, file))

# 3. Create the dataset.yaml file
dataset_config = {
    'path': os.path.abspath(DATA_DIR), # Absolute path to the data directory
    'train': 'images/train',           # Relative path to training images
    'val': 'images/train',             # Using train for val in this quick test
    'names': {
        0: 'tube'
    }
}


with open('dataset.yaml', 'w') as f:
    yaml.dump(dataset_config, f, default_flow_style=False, sort_keys=False)

print("dataset.yaml created successfully! You can now run the training script.")


# Load a pretrained lightweight model
model = YOLO('yolov8n.pt') 

# Train the model
results = model.train(
    data='dataset.yaml', 
    epochs=20, 
    imgsz=640, 
    batch=8,
    project='tube_detection',
    name='yolo_model'
)

Organizing files for YOLO...
dataset.yaml created successfully! You can now run the training script.
New https://pypi.org/project/ultralytics/8.4.51 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.48  Python-3.13.13 torch-2.11.0+cpu CPU (13th Gen Intel Core i7-1355U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0

In [7]:
import cv2
import math
import numpy as np
from ultralytics import YOLO

# --- Configuration ---
TRAINED_MODEL_PATH = r'runs\detect\tube_detection\yolo_model-5\weights\best.pt'
TEST_IMAGE_PATH    = r"images\71a6769b-color.png" # --- for testing an image
CONF_THRESHOLD     = 0.5   # Minimum confidence to display a detection

# --- Load the trained model ---
model = YOLO(TRAINED_MODEL_PATH)

# --- Run Inference ---
img = cv2.imread(TEST_IMAGE_PATH)
if img is None:
    raise FileNotFoundError(f"Could not read image: {TEST_IMAGE_PATH}")

h, w = img.shape[:2]

results = model.predict(
    source=TEST_IMAGE_PATH,
    conf=CONF_THRESHOLD,
    save=False,      # We'll draw manually for full control
    verbose=True
)

# --- Parse & Visualize Detections ---
result = results[0]   # Single image → first (only) result
boxes  = result.boxes  # Boxes object

print(f"\nDetected {len(boxes)} tube(s):\n")

for i, box in enumerate(boxes):
    # Bounding box in pixel coords (xyxy format)
    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
    conf            = float(box.conf[0])
    cls             = int(box.cls[0])
    label           = f"tube {conf:.2f}"

    # Derived geometry
    cx = (x1 + x2) // 2
    cy = (y1 + y2) // 2
    bw = x2 - x1
    bh = y2 - y1

    print(f"  [{i+1}] class={cls}  conf={conf:.3f}  "
          f"box=({x1},{y1},{x2},{y2})  center=({cx},{cy})  "
          f"size=({bw}x{bh})")




image 1/1 e:\Zeon_Systems\TubeDetectionDataset_ZeonSystems2026\old\images\71a6769b-color.png: 480x640 4 tubes, 188.6ms
Speed: 27.8ms preprocess, 188.6ms inference, 3.0ms postprocess per image at shape (1, 3, 480, 640)

Detected 4 tube(s):

  [1] class=0  conf=0.879  box=(489,48,544,108)  center=(516,78)  size=(55x60)
  [2] class=0  conf=0.807  box=(425,126,479,176)  center=(452,151)  size=(54x50)
  [3] class=0  conf=0.738  box=(456,87,509,144)  center=(482,115)  size=(53x57)
  [4] class=0  conf=0.610  box=(396,159,442,215)  center=(419,187)  size=(46x56)


In [32]:
import os, math, random
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

# ── Model ────────────────────────────────────────────────────────────
class TabAngle1DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=15, padding=7),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=15, padding=7),
            nn.ReLU(),
            nn.Conv1d(32, 16, kernel_size=15, padding=7),
            nn.ReLU(),
        )
        self.fc = nn.Linear(16 * 360, 2)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

# ── Profile Extractor ─────────────────────────────────────────────────
def extract_profile(img_path, cx, cy, n_angles=360):
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    radius = min(h, w) // 3

    profile = []
    for i in range(n_angles):
        angle = 2 * math.pi * i / n_angles
        vals = []
        for dr in range(-5, 10):
            r = radius + dr
            px = int(cx + r * math.cos(angle))
            py = int(cy - r * math.sin(angle))
            if 0 <= px < w and 0 <= py < h:
                vals.append(float(gray[py, px]))
        profile.append(np.mean(vals) if vals else 0)

    profile = np.array(profile, dtype=np.float32)
    # Normalize
    profile = (profile - profile.mean()) / (profile.std() + 1e-6)
    return profile

# ── Dataset ───────────────────────────────────────────────────────────
class ProfileDataset(Dataset):
    def __init__(self, csv_path, crops_dir, augment=True):
        self.df = pd.read_csv(csv_path)
        self.crops_dir = crops_dir
        self.augment = augment

        # Pre-extract all profiles (fast, do once)
        print("Extracting radial profiles...")
        self.profiles = []
        self.angles = []
        for _, row in self.df.iterrows():
            img_path = os.path.join(crops_dir, row['crop_name'])
            # cx, cy are lid center within the crop
            cx = row.get('crop_cx', None) or (cv2.imread(img_path).shape[1] // 2)
            cy = row.get('crop_cy', None) or (cv2.imread(img_path).shape[0] // 2)
            profile = extract_profile(img_path, cx, cy)
            self.profiles.append(profile)
            self.angles.append(float(row['angle_deg']))
        print(f"Done. {len(self.profiles)} profiles extracted.")

    def __len__(self):
        return len(self.profiles)

    def __getitem__(self, idx):
        profile = self.profiles[idx].copy()
        angle_deg = self.angles[idx]

        if self.augment:
            # Exact rotation augmentation — just shift the array!
            shift = random.randint(0, 359)
            profile = np.roll(profile, shift)
            angle_deg = (angle_deg - shift + 360) % 360

        angle_rad = math.radians(angle_deg)
        target = torch.tensor([math.cos(angle_rad),
                                math.sin(angle_rad)], dtype=torch.float32)
        x = torch.tensor(profile, dtype=torch.float32).unsqueeze(0)  # (1, 360)
        return x, target

# ── Training ──────────────────────────────────────────────────────────
df = pd.read_csv('data/crop_annotations.csv')
full_ds = ProfileDataset('data/crop_annotations.csv', 'data/crops', augment=True)

val_size = max(1, int(0.15 * len(full_ds)))
train_ds, val_ds = random_split(full_ds, [len(full_ds) - val_size, val_size])
val_ds.dataset.augment = False  # no augmentation for val

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TabAngle1DCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
criterion = nn.MSELoss()

best_mae = float('inf')
for epoch in range(50):
    # Train
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()

    # Validate
    model.eval()
    errors = []
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x).cpu().numpy()
            gt   = y.cpu().numpy()
            for p, g in zip(pred, gt):
                p_deg = (math.degrees(math.atan2(p[1], p[0])) + 360) % 360
                g_deg = (math.degrees(math.atan2(g[1], g[0])) + 360) % 360
                err = abs(p_deg - g_deg)
                errors.append(min(err, 360 - err))

    mae = np.mean(errors)
    scheduler.step()
    print(f"Epoch {epoch+1:02d} | MAE: {mae:.2f}°")

    if mae < best_mae:
        best_mae = mae
        torch.save(model.state_dict(), 'tab_angle_1dcnn_best.pth')
        print(f"  ✓ Saved (MAE: {best_mae:.2f}°)")

print(f"\nBest MAE: {best_mae:.2f}°")

Extracting radial profiles...
Done. 371 profiles extracted.
Epoch 01 | MAE: 71.25°
  ✓ Saved (MAE: 71.25°)
Epoch 02 | MAE: 60.71°
  ✓ Saved (MAE: 60.71°)
Epoch 03 | MAE: 58.97°
  ✓ Saved (MAE: 58.97°)
Epoch 04 | MAE: 45.76°
  ✓ Saved (MAE: 45.76°)
Epoch 05 | MAE: 40.24°
  ✓ Saved (MAE: 40.24°)
Epoch 06 | MAE: 45.97°
Epoch 07 | MAE: 42.55°
Epoch 08 | MAE: 46.47°
Epoch 09 | MAE: 38.12°
  ✓ Saved (MAE: 38.12°)
Epoch 10 | MAE: 43.96°
Epoch 11 | MAE: 38.02°
  ✓ Saved (MAE: 38.02°)
Epoch 12 | MAE: 33.99°
  ✓ Saved (MAE: 33.99°)
Epoch 13 | MAE: 37.82°
Epoch 14 | MAE: 34.94°
Epoch 15 | MAE: 35.11°
Epoch 16 | MAE: 38.51°
Epoch 17 | MAE: 30.74°
  ✓ Saved (MAE: 30.74°)
Epoch 18 | MAE: 29.66°
  ✓ Saved (MAE: 29.66°)
Epoch 19 | MAE: 36.34°
Epoch 20 | MAE: 30.75°
Epoch 21 | MAE: 30.01°
Epoch 22 | MAE: 29.14°
  ✓ Saved (MAE: 29.14°)
Epoch 23 | MAE: 30.00°
Epoch 24 | MAE: 28.54°
  ✓ Saved (MAE: 28.54°)
Epoch 25 | MAE: 30.91°
Epoch 26 | MAE: 27.84°
  ✓ Saved (MAE: 27.84°)
Epoch 27 | MAE: 28.94°
Epoch 2

In [33]:
import os
import cv2
import pandas as pd
import numpy as np
import math
import torch
import torch.nn as nn
from ultralytics import YOLO
from scipy.optimize import linear_sum_assignment

# ==========================================
# 1. Configuration
# ==========================================
CSV_PATH = 'annotations.csv'
IMAGES_DIR = 'images'

YOLO_WEIGHTS_PATH = r'E:\Zeon_Systems\TubeDetectionDataset_ZeonSystems2026\runs\detect\tube_detection\yolo_model-5\weights\best.pt'
CNN1D_WEIGHTS_PATH = 'tab_angle_1dcnn_best.pth'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running evaluation on: {device}")

# ==========================================
# 2. 1D CNN Model Definition
#    (must match training script exactly)
# ==========================================
class TabAngle1DCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=15, padding=7),
            nn.ReLU(),
            nn.Conv1d(16, 32, kernel_size=15, padding=7),
            nn.ReLU(),
            nn.Conv1d(32, 16, kernel_size=15, padding=7),
            nn.ReLU(),
        )
        self.fc = nn.Linear(16 * 360, 2)

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)

# ==========================================
# 3. Radial Profile Extractor
#    (must match training script exactly)
# ==========================================
def extract_profile(crop, cx, cy, n_angles=360):
    """
    crop  : BGR numpy array (the YOLO crop)
    cx,cy : lid center coordinates WITHIN the crop
    """
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    radius = min(h, w) // 3

    profile = []
    for i in range(n_angles):
        angle = 2 * math.pi * i / n_angles
        vals = []
        for dr in range(-5, 10):
            r = radius + dr
            px = int(cx + r * math.cos(angle))
            py = int(cy - r * math.sin(angle))
            if 0 <= px < w and 0 <= py < h:
                vals.append(float(gray[py, px]))
        profile.append(np.mean(vals) if vals else 0.0)

    profile = np.array(profile, dtype=np.float32)
    # Normalize — same as training
    profile = (profile - profile.mean()) / (profile.std() + 1e-6)
    return profile

# ==========================================
# 4. Load Models
# ==========================================
print("Loading models...")

yolo_model = YOLO(YOLO_WEIGHTS_PATH)

angle_model = TabAngle1DCNN().to(device)
angle_model.load_state_dict(torch.load(CNN1D_WEIGHTS_PATH, map_location=device))
angle_model.eval()

print("Models loaded.")

# ==========================================
# 5. Evaluation Logic
# ==========================================
def evaluate_pipeline(csv_path, images_dir):
    df = pd.read_csv(csv_path)

    true_positives  = 0
    false_positives = 0
    false_negatives = 0

    angle_errors  = []
    center_errors = []

    total_gt    = 0
    total_preds = 0

    DISTANCE_THRESHOLD = 30

    print("Evaluating images...")

    for img_name in df['image'].unique():
        img_path = os.path.join(images_dir, img_name)
        img = cv2.imread(img_path)
        if img is None:
            print(f"Warning: Could not load {img_name}")
            continue

        # Ground truth
        gt_subset  = df[df['image'] == img_name]
        gt_centers = gt_subset[['center_x', 'center_y']].values
        gt_angles  = gt_subset['angle_deg'].values
        total_gt  += len(gt_centers)

        # YOLO predictions
        results     = yolo_model(img, verbose=False)[0]
        boxes       = results.boxes.xyxy.cpu().numpy()
        total_preds += len(boxes)

        pred_centers = []
        pred_angles  = []

        for box in boxes:
            x1, y1, x2, y2 = map(int, box[:4])
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            pred_centers.append([cx, cy])

            # Crop
            crop = img[max(0, y1):min(img.shape[0], y2),
                       max(0, x1):min(img.shape[1], x2)]
            if crop.size == 0:
                pred_angles.append(0.0)
                continue

            # Lid center WITHIN the crop
            crop_cx = cx - x1
            crop_cy = cy - y1

            # Extract radial profile
            profile  = extract_profile(crop, crop_cx, crop_cy)
            x_tensor = torch.tensor(profile).unsqueeze(0).unsqueeze(0).to(device)  # (1,1,360)

            # 1D CNN inference
            with torch.no_grad():
                out = angle_model(x_tensor).cpu().numpy()[0]

            pred_angle_deg = (math.degrees(math.atan2(out[1], out[0])) + 360) % 360
            pred_angles.append(pred_angle_deg)

        pred_centers = np.array(pred_centers)

        # Hungarian matching
        if len(pred_centers) == 0:
            false_negatives += len(gt_centers)
            continue

        cost_matrix      = np.linalg.norm(gt_centers[:, np.newaxis] - pred_centers, axis=2)
        row_ind, col_ind = linear_sum_assignment(cost_matrix)

        matched_gt   = set()
        matched_pred = set()

        for r, c in zip(row_ind, col_ind):
            pixel_distance = cost_matrix[r, c]
            if pixel_distance < DISTANCE_THRESHOLD:
                true_positives += 1
                matched_gt.add(r)
                matched_pred.add(c)

                center_errors.append(pixel_distance)

                err          = abs(gt_angles[r] - pred_angles[c])
                circular_err = min(err, 360 - err)
                angle_errors.append(circular_err)

        false_negatives += len(gt_centers)   - len(matched_gt)
        false_positives += len(pred_centers) - len(matched_pred)

        # ==========================================
        # Save Visualized Output Images
        # ==========================================
        VIS_DIR = 'visualizations'
        os.makedirs(VIS_DIR, exist_ok=True)

        # Draw GT (green) and Predictions (red) on image
        vis = img.copy()

        # Draw Ground Truth
        for i, (gcx, gcy) in enumerate(gt_centers):
            gcx, gcy = int(gcx), int(gcy)
            g_angle  = gt_angles[i]
            g_rad    = math.radians(g_angle)
            arrow_len = 30
            gax = int(gcx + arrow_len * math.cos(g_rad))
            gay = int(gcy - arrow_len * math.sin(g_rad))  # flip Y for CCW

            cv2.circle(vis, (gcx, gcy), 8, (0, 255, 0), 2)           # green circle = GT center
            cv2.arrowedLine(vis, (gcx, gcy), (gax, gay),
                            (0, 255, 0), 2, tipLength=0.3)            # green arrow = GT angle
            cv2.putText(vis, f"GT:{g_angle:.0f}", (gcx + 10, gcy - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)

        # Draw Predictions
        for j, (pcx, pcy) in enumerate(pred_centers):
            pcx, pcy  = int(pcx), int(pcy)
            p_angle   = pred_angles[j]
            p_rad     = math.radians(p_angle)
            arrow_len = 30
            pax = int(pcx + arrow_len * math.cos(p_rad))
            pay = int(pcy - arrow_len * math.sin(p_rad))

            cv2.circle(vis, (pcx, pcy), 8, (0, 0, 255), 2)           # red circle = pred center
            cv2.arrowedLine(vis, (pcx, pcy), (pax, pay),
                            (0, 0, 255), 2, tipLength=0.3)            # red arrow = pred angle
            cv2.putText(vis, f"PR:{p_angle:.0f}", (pcx + 10, pcy + 15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 255), 1)

        # Draw matched pairs — show angle error
        for r, c in zip(row_ind, col_ind):
            if cost_matrix[r, c] < DISTANCE_THRESHOLD:
                err          = abs(gt_angles[r] - pred_angles[c])
                circular_err = min(err, 360 - err)
                gcx, gcy     = int(gt_centers[r][0]), int(gt_centers[r][1])

                # Color code: green < 15°, yellow < 30°, red >= 30°
                if circular_err <= 15:
                    color = (0, 255, 0)
                elif circular_err <= 30:
                    color = (0, 255, 255)
                else:
                    color = (0, 0, 255)

                cv2.putText(vis, f"err:{circular_err:.0f}",
                            (gcx - 30, gcy + 30),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.4, color, 1)

        # Legend
        cv2.putText(vis, "Green=GT  Red=Pred", (10, 20),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        cv2.imwrite(os.path.join(VIS_DIR, img_name), vis)

    # ==========================================
    # 6. Final Metrics
    # ==========================================
    precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) > 0 else 0
    recall    = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) > 0 else 0
    f1_score  = 2 * (precision * recall) / (precision + recall)    if (precision + recall) > 0            else 0

    mae_angle    = np.mean(angle_errors)                                          if angle_errors else 0
    median_angle = np.median(angle_errors)                                        if angle_errors else 0
    acc_15_deg   = sum(1 for e in angle_errors if e <= 15) / len(angle_errors)   if angle_errors else 0
    mae_center   = np.mean(center_errors)                                         if center_errors else 0

    print("\n=============== FINAL RESULTS ===============")
    print(f"Total Ground Truth:  {total_gt}")
    print(f"Total Predictions:   {total_preds}")
    print(f"True Positives:      {true_positives}")
    print(f"False Positives:     {false_positives}")
    print(f"False Negatives:     {false_negatives}")
    print("---------------------------------------------")
    print(f"Precision:           {precision:.4f}")
    print(f"Recall:              {recall:.4f}")
    print(f"F1-Score:            {f1_score:.4f}")
    print("---------------------------------------------")
    print(f"Center MAE (Mean):   {mae_center:.2f} pixels")
    print(f"Angle MAE (Mean):    {mae_angle:.2f}°")
    print(f"Angle Median Error:  {median_angle:.2f}°")
    print(f"Angle Acc (<= 15°):  {acc_15_deg * 100:.1f}%")
    print("=============================================")

if __name__ == '__main__':
    evaluate_pipeline(CSV_PATH, IMAGES_DIR)

Running evaluation on: cpu
Loading models...
Models loaded.
Evaluating images...

=============== FINAL RESULTS ===============
Total Ground Truth:  371
Total Predictions:   375
True Positives:      371
False Positives:     4
False Negatives:     0
---------------------------------------------
Precision:           0.9893
Recall:              1.0000
F1-Score:            0.9946
---------------------------------------------
Center MAE (Mean):   1.19 pixels
Angle MAE (Mean):    27.33°
Angle Median Error:  18.87°
Angle Acc (<= 15°):  40.7%
